## Gold層 集計やドメイン特化型のデータを作成
### 概要
クレンジング済みのSilver層データをもとに、集計を行ったりそれぞれの分析に特化したデータを作成する。

In [0]:
from pyspark.sql.functions import col, count, avg, max, min, round, current_timestamp

# 1. カタログとスキーマの設定
spark.sql("USE CATALOG my_ecommerce_pipeline")
spark.sql("USE SCHEMA gold")

# 2. Silver層からのデータ読み込み
silver_df = spark.read.table("silver.cleaned_diamonds")

# 3. 集計・KPI算出(データマートの作成)
# カット(cut)とカラー(color)別の分析用集計
gold_aggregated_df = (
    silver_df
    .groupBy("cut", "color")
    .agg(
        count("*").alias("total_diamonds"),
        round(avg("price"), 2).alias("avg_price"),
        round(avg("carat"), 2).alias("avg_carat"),
        # カラット当たりの単価平均(price / carat)
        round(avg(col("price") / col("carat")), 2).alias("avg_price_per_carat"),
        min("price").alias("min_price"),
        max("price").alias("max_price")
    )
    .withColumn("_created_at", current_timestamp())
)

# 4. Gold層への書き込み(Overwriteモード)
# 集計マートは最新のSilverデータ全体から再計算して置き換える(Overwrite)ことで冪等性を保持
# ()内に処理を書いているのは、複数行に綺麗に改行して処理の流れを分かりやすくするため
(
    gold_aggregated_df
    .write
    .format("delta")
    .mode("overwrite") # 更新するたびに全件上書き
    .option("overwriteSchema", "true")
    .saveAsTable("gold.diamond_summary_by_cut_color")
)

print("Gold層(集計マート)の作成が完了しました!")